# 🕵️ Football Lab 2: Stat Detective
You're a detective, and your evidence is **every NFL regular season game from 1999 to 2024** — almost 130,000 player stat lines.
Your job: crack the cases using Python.

**How it works:**
- Run a gray box with **Shift + Enter**.
- `___` (three underscores) means **you fill it in**. The code won't work until you do!
- 🔎 **Google it** gives you words to search when you get stuck. Googling is not cheating —
  real programmers google things all day long.
- Got a red error? Read the last line. It usually tells you what's wrong.

---
## Case 1. Open the evidence box 📦

In [ ]:
import pandas as pd

stats = pd.read_csv(
    "https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats.csv",
    low_memory=False,
)
stats = stats[stats["season_type"] == "REG"]   # regular season only, no playoffs
stats.head()

Each row = **one player in one game**. How many rows are there?

🔎 **Google it:** `python how to count length of a list` (the same trick works for tables)

In [ ]:
print("Rows:", ___(stats))

How many columns does the table have, and what are they called?

🔎 **Google it:** `pandas show column names`

In [ ]:
stats.___

---
## Case 2. Track down your player 🎯
Let's find every game your favorite player played. Type his name exactly, like `"Travis Kelce"`.

In [ ]:
name = "___"

games = stats[stats["player_display_name"] == name]
print(name, "played", len(games), "regular season games")
games[["season", "week", "recent_team", "opponent_team", "receiving_yards", "rushing_yards", "passing_yards"]].head(10)

What was his **best** game ever? Find the biggest number in a column.

🔎 **Google it:** `pandas max value of column`

In [ ]:
stat = "receiving_yards"   # QB? use "passing_yards". RB? use "rushing_yards"

print("Best game:", games[stat].___())

And how many yards does he get **on average** per game?

🔎 **Google it:** `pandas average of column`

In [ ]:
print("Average per game:", round(games[stat].___(), 1))

---
## Case 3. The greatest single games ever 💥
Sort every game ever by receiving yards, biggest first.

In [ ]:
biggest = stats.sort_values("receiving_yards", ascending=False)
biggest[["player_display_name", "season", "week", "recent_team", "opponent_team", "receiving_yards"]].head(5)

Who's #1? Do you know that nickname? 🤖

### 🏈 Your turn
Now find the **biggest passing game** ever. Copy the code above and change the column name.
Then try `"rushing_yards"` and `"passing_tds"`.

In [ ]:
biggest = stats.sort_values("___", ascending=False)
biggest[["player_display_name", "season", "week", "recent_team", "opponent_team", "___"]].head(5)

---
## Case 4. Best season ever 📅
To get a whole season, we add up all the games for each player in each year.
That's called **grouping**.

In [ ]:
seasons = stats.groupby(["player_display_name", "season"])["receiving_yards"].sum()
seasons.sort_values(ascending=False).head(5)

### 🚨 Something's fishy!
Look at #1. **Steve Smith, 2,202 yards in 2009?** That would be the all-time record by a mile.
The real record is Calvin Johnson's 1,964 in 2012. So what happened?

**Clue:** in 2009 there were **two** different receivers named Steve Smith —
one on the Panthers (CAR) and one on the Giants (NYG). Let's check:

In [ ]:
steve = stats[(stats["player_display_name"] == "Steve Smith") & (stats["season"] == 2009)]
steve.groupby(["player_id", "recent_team"])["receiving_yards"].sum()

Python glued two people together because they had the same name! 😱
Every player has his own `player_id`, like a fingerprint. Let's group by that too, and the case is solved:

In [ ]:
seasons = stats.groupby(["player_id", "player_display_name", "season"])["receiving_yards"].sum()
seasons.sort_values(ascending=False).head(5)

### 🏈 Your turn
Find the best seasons ever for **passing touchdowns** (`"passing_tds"`) and **rushing yards** (`"rushing_yards"`).

In [ ]:
seasons = stats.groupby(["player_id", "player_display_name", "season"])["___"].sum()
seasons.sort_values(ascending=False).head(5)

---
## Case 5. Head to head ⚔️
Let's write a **function** that compares two players in any stat over their whole careers.

In [ ]:
def career(name, stat):
    return stats[stats["player_display_name"] == name][stat].sum()


def compare(player1, player2, stat):
    a = career(player1, stat)
    b = career(player2, stat)
    print(player1, ":", a)
    print(player2, ":", b)
    if a > b:
        print("Winner:", player1, "🏆")
    else:
        print("Winner:", player2, "🏆")


compare("Tom Brady", "Peyton Manning", "passing_tds")

### 🏈 Your turn
Settle an argument! Pick two players and a stat. Maybe Justin Jefferson vs Ja'Marr Chase in `"receiving_yards"`?

**Bonus:** what happens if both have the same number? Can you add a `"It's a tie!"` message?
🔎 **Google it:** `python elif`

In [ ]:
compare("___", "___", "___")

---
## Case 6. Team detective 🏟️
Which team scored the most passing touchdowns in one season?
`recent_team` is the team, and we group by team + season.

In [ ]:
team_seasons = stats.groupby(["recent_team", "season"])["passing_tds"].sum()
team_seasons.sort_values(ascending=False).head(5)

### 🏈 Your turn
Now find out how many receiving yards **your favorite team** got in each season.
First pick only your team's rows, then group by season.

🔎 **Google it:** `pandas filter rows by column value`

In [ ]:
my_team = "___"   # like "KC", "PHI", "WAS", "MIN"

team = stats[stats["recent_team"] == my_team]
team.groupby("season")["receiving_yards"].___()

---
## 🔍 Final case: the mystery player
Use the clues to find him. Each clue is a filter. Keep adding filters until only **one** player is left.

**Clues:**
1. He's a quarterback (`position` is `"QB"`).
2. In one season he threw for **more than 5,000 yards**.
3. He did it **after 2015**.
4. His team was **KC**.

In [ ]:
qb = stats[stats["position"] == "QB"]
qb_seasons = qb.groupby(["player_id", "player_display_name", "recent_team", "season"])["passing_yards"].sum().reset_index()

suspects = qb_seasons[qb_seasons["passing_yards"] > 5000]
suspects

Lots of suspects! Add the other two clues. Hint: you can join filters with `&`,
and each filter needs its own `( )` parentheses.

In [ ]:
suspects = qb_seasons[(qb_seasons["passing_yards"] > 5000) & (qb_seasons["season"] > ___) & (qb_seasons["recent_team"] == "___")]
suspects

### 🏈 Your turn: make your own mystery
Make up 3–4 clues about a player you know, write the code, and give the clues to mom or dad.
Can they find him? (No Googling allowed for them! 😄)

In [ ]:
# your mystery player

---
### 🏆 Case closed, Detective!
New skills unlocked: `len`, `.max()`, `.mean()`, `.sum()`, `sort_values`, `groupby`, filters with `&`, and functions.
That's real data analysis. Next time you argue about who's better, you can **prove** it.